<a href="https://colab.research.google.com/github/fourmodern/2025_aidrugdiscovery/blob/main/qwen_tcga_advanced_2024.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Advanced Qwen Training with ORPO/DPO on TCGA Dataset

최신 alignment 기법을 사용한 고급 훈련 방법입니다.

## 1. Install Latest Packages

In [ ]:
# Latest alignment training packages
#
# 주의 1) !pip install pkg>=1.0 처럼 쓰면 셸이 ">"를 리다이렉션으로 해석해서
#         버전 조건이 무시되고 "=1.0" 이라는 빈 파일이 생깁니다.
#         버전 조건은 반드시 따옴표로 묶어야 합니다.
# 주의 2) torch는 Colab에 이미 설치되어 있어 재설치하지 않습니다.
#         (재설치하면 런타임의 CUDA 빌드와 어긋날 수 있습니다)
!pip install -q "transformers>=4.56"
!pip install -q trl              # DPO/KTO/GRPO
!pip install -q datasets accelerate
!pip install -q "peft>=0.9.0"
!pip install -q bitsandbytes     # Optional for memory

# flash-attn: GPU Ampere+ (compute capability >= 8.0) 필요
import subprocess
try:
    subprocess.check_call("pip install -q flash-attn --no-build-isolation".split())
    print("flash-attn installed")
except Exception:
    print("flash-attn unavailable - using default attention (slower but functional)")
!pip install -q wandb  # For tracking

# ORPO에 대한 안내
# TRL 1.x에서 ORPOTrainer / ORPOConfig 가 upstream에서 제거되었습니다.
# (DPOConfig의 loss_type 목록에도 'orpo'는 없습니다)
# ORPO 셀을 그대로 실행하려면 아래처럼 예전 TRL을 설치해야 합니다:
#     !pip install -q "trl==0.24.0"
# 단, 그 버전은 최신 transformers와 조합할 때 DPO 경로가
# 옵션 의존성(mergekit, llm_blender) 문제로 import되지 않습니다.
# 기본 경로로는 DPO를 사용하시길 권합니다.
import trl
print(f"✅ Latest packages installed (trl {trl.__version__})")
print("   ORPO 사용 가능:", hasattr(trl, "ORPOTrainer"))

## 2. Environment Setup

In [ ]:
import torch
import os
from typing import Dict, List, Optional, Tuple
import warnings
warnings.filterwarnings('ignore')

# GPU check
gpu_info = torch.cuda.get_device_properties(0)
print(f"🚀 GPU: {torch.cuda.get_device_name(0)}")
print(f"💾 VRAM: {gpu_info.total_memory / 1024**3:.1f} GB")
print(f"🔥 Flash Attention: {torch.cuda.get_device_capability()[0] >= 8}")

# Optimal settings based on GPU
# 모델은 Qwen3 계열로 갱신했습니다 (Qwen2.5 -> Qwen3)
if "A100" in torch.cuda.get_device_name(0):
    MODEL_ID = "Qwen/Qwen3-14B"  # Larger model
    USE_FLASH_ATTN = True
    USE_QUANTIZATION = False  # No quantization for better quality
    BATCH_SIZE = 4
elif "H100" in torch.cuda.get_device_name(0):
    MODEL_ID = "Qwen/Qwen3-32B"  # Largest model
    USE_FLASH_ATTN = True
    USE_QUANTIZATION = False
    BATCH_SIZE = 2
else:
    MODEL_ID = "Qwen/Qwen3-8B"
    USE_FLASH_ATTN = False
    USE_QUANTIZATION = True  # Only for low memory
    BATCH_SIZE = 1

print(f"\n📋 Configuration:")
print(f"  Model: {MODEL_ID}")
print(f"  Flash Attention: {USE_FLASH_ATTN}")
print(f"  Quantization: {USE_QUANTIZATION}")

# Mount Drive
from google.colab import drive
drive.mount('/content/drive')

In [9]:
import os
import torch
import gc

# 1. 메모리 정리
torch.cuda.empty_cache()
gc.collect()
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

In [10]:
# 3. 설정 수정
config.batch_size = 1
config.gradient_accumulation = 16
config.use_lora = True  # LoRA 활성화
config.lora_r = 32  # 작은 rank
config.max_length = 768  # 길이 줄이기

## 3. Advanced Configuration

In [3]:
from dataclasses import dataclass
from enum import Enum

class TrainingMethod(Enum):
    SFT = "sft"  # Supervised Fine-Tuning
    DPO = "dpo"  # Direct Preference Optimization
    ORPO = "orpo"  # Odds Ratio Preference Optimization
    KTO = "kto"  # Kahneman-Tversky Optimization
    GRPO = "grpo"  # Group Relative Policy Optimization

@dataclass
class TrainingConfig:
    # Method
    method: TrainingMethod = TrainingMethod.ORPO  # Most advanced

    # Model
    model_id: str = MODEL_ID
    use_flash_attention: bool = USE_FLASH_ATTN
    use_quantization: bool = USE_QUANTIZATION

    # Data
    train_data: str = "/content/drive/MyDrive/tcga_qa/train_300k_conversation.jsonl"
    test_data: str = "/content/drive/MyDrive/tcga_qa/test_300k_conversation.jsonl"

    # Training
    batch_size: int = BATCH_SIZE
    gradient_accumulation: int = 8 // BATCH_SIZE
    learning_rate: float = 5e-5
    num_epochs: int = 2
    max_steps: int = -1  # Use all data

    # ORPO/DPO specific
    beta: float = 0.1  # Preference strength
    max_prompt_length: int = 512
    max_length: int = 1024

    # LoRA (optional)
    use_lora: bool = False  # Full fine-tuning for best quality
    lora_r: int = 128
    lora_alpha: int = 256

    # Output
    output_dir: str = f"/content/drive/MyDrive/tcga_qa/{method.value}_model"

config = TrainingConfig()
print(f"🎯 Training Method: {config.method.value.upper()}")
print(f"🔧 Full Fine-tuning: {not config.use_lora}")

🎯 Training Method: ORPO
🔧 Full Fine-tuning: True


## 4. Load Model (No Quantization for Quality)

In [4]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

print(f"🤖 Loading {config.model_id}...\n")

# Model loading arguments
model_kwargs = {
    "torch_dtype": torch.bfloat16,  # BF16 for training stability
    "device_map": "auto",
    "trust_remote_code": True,
}

# Add Flash Attention if supported
if config.use_flash_attention:
    model_kwargs["attn_implementation"] = "flash_attention_2"
    print("⚡ Using Flash Attention 2")

# Optional quantization (only if necessary)
if config.use_quantization:
    from transformers import BitsAndBytesConfig
    model_kwargs["quantization_config"] = BitsAndBytesConfig(
        load_in_8bit=True,  # 8-bit is better than 4-bit
        bnb_8bit_compute_dtype=torch.bfloat16,
    )
    print("📉 Using 8-bit quantization (not recommended)")

# Load model
model = AutoModelForCausalLM.from_pretrained(
    config.model_id,
    **model_kwargs
)

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(
    config.model_id,
    trust_remote_code=True,
)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "left"  # For generation

print("✅ Model loaded successfully")

# Enable gradient checkpointing for memory
model.gradient_checkpointing_enable()
model.config.use_cache = False

# Optional: LoRA for memory constraints
if config.use_lora:
    from peft import LoraConfig, get_peft_model, TaskType

    peft_config = LoraConfig(
        r=config.lora_r,
        lora_alpha=config.lora_alpha,
        lora_dropout=0.05,
        bias="none",
        task_type=TaskType.CAUSAL_LM,
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                       "gate_proj", "up_proj", "down_proj"],
    )
    model = get_peft_model(model, peft_config)
    model.print_trainable_parameters()
else:
    print("🎯 Full model fine-tuning (best quality)")

🤖 Loading Qwen/Qwen2.5-14B-Instruct...

⚡ Using Flash Attention 2


config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 8 files:   0%|          | 0/8 [00:00<?, ?it/s]

...

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

✅ Model loaded successfully
🎯 Full model fine-tuning (best quality)


## 5. Prepare Preference Dataset for ORPO/DPO

In [5]:
import json
from datasets import Dataset
from tqdm.auto import tqdm
import random

def create_preference_pairs(data_path, max_samples=None):
    """Create preference pairs for ORPO/DPO training"""
    preference_data = []

    with open(data_path, 'r') as f:
        data = [json.loads(line) for line in f]

    if max_samples:
        data = data[:max_samples]

    for item in tqdm(data, desc="Creating preference pairs"):
        messages = item.get('messages', [])

        if len(messages) >= 3:
            system = messages[0]['content']
            question = messages[1]['content']
            correct_answer = messages[2]['content']

            # Create prompt
            prompt = f"{system}\n\nQuestion: {question}\nAnswer:"

            # Create rejected answers (wrong answers)
            rejected_answers = []

            # Type 1: Opposite answer for Yes/No questions
            if correct_answer.lower() in ['yes', 'no']:
                rejected = 'No' if correct_answer.lower() == 'yes' else 'Yes'
                rejected_answers.append(rejected)

            # Type 2: Random wrong answer
            wrong_answers = [
                "I don't have enough information to answer this question.",
                "This is unclear from the data.",
                "The evidence is inconclusive.",
                "Further analysis is needed.",
            ]
            rejected_answers.append(random.choice(wrong_answers))

            # Type 3: Slightly wrong answer (if contains gene names)
            if any(gene in correct_answer for gene in ['TP53', 'BRCA', 'EGFR', 'MYC']):
                wrong_genes = ['KRAS', 'BRAF', 'PIK3CA', 'PTEN', 'APC']
                for gene in wrong_genes:
                    if gene not in correct_answer:
                        rejected_answers.append(gene)
                        break

            # Add to dataset
            for rejected in rejected_answers[:1]:  # Use best rejected
                preference_data.append({
                    "prompt": prompt,
                    "chosen": correct_answer,
                    "rejected": rejected,
                })

    return preference_data

print("📚 Creating preference dataset...\n")

# Create training preference pairs
train_preferences = create_preference_pairs(config.train_data)
train_dataset = Dataset.from_list(train_preferences)

# Create test preference pairs
test_preferences = create_preference_pairs(config.test_data, max_samples=1000)
test_dataset = Dataset.from_list(test_preferences)

print(f"\n✅ Preference datasets created:")
print(f"  Training: {len(train_dataset):,} preference pairs")
print(f"  Test: {len(test_dataset):,} preference pairs")

# Show example
example = train_dataset[0]
print(f"\n📝 Example preference pair:")
print(f"  Prompt: {example['prompt'][:100]}...")
print(f"  Chosen: {example['chosen']}")
print(f"  Rejected: {example['rejected']}")

📚 Creating preference dataset...



Creating preference pairs:   0%|          | 0/101314 [00:00<?, ?it/s]

Creating preference pairs:   0%|          | 0/1000 [00:00<?, ?it/s]


✅ Preference datasets created:
  Training: 101,314 preference pairs
  Test: 1,000 preference pairs

📝 Example preference pair:
  Prompt: You are an expert in cancer genomics, specifically in analyzing differential gene expression pattern...
  Chosen: No
  Rejected: Yes


## 6. Setup ORPO Training (Latest Method)

In [14]:
from trl import ORPOConfig, ORPOTrainer

if config.method == TrainingMethod.ORPO:
    # ORPO Configuration
    training_args = ORPOConfig(
        output_dir=config.output_dir,

        # Training parameters
        num_train_epochs=config.num_epochs,
        per_device_train_batch_size=config.batch_size,
        per_device_eval_batch_size=config.batch_size,
        gradient_accumulation_steps=config.gradient_accumulation,
        gradient_checkpointing=True,

        # ORPO specific
        beta=config.beta,  # Lower beta = more focus on SFT
        learning_rate=config.learning_rate,
        lr_scheduler_type="cosine",
        warmup_ratio=0.1,
        optim="paged_adamw_8bit",
        # optim="adamw_torch",  # Standard optimizer

        # Logging
        logging_steps=25,
        save_steps=500,
        eval_steps=500,

        # Performance
        bf16=True,

        # Limits
        max_length=config.max_length,
        max_prompt_length=config.max_prompt_length,

        # Other
        remove_unused_columns=False,
        report_to="none",  # or "wandb"
    )

    # Initialize ORPO Trainer - v0.12+ API
    trainer = ORPOTrainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=test_dataset,
        processing_class=tokenizer,  # 변경: tokenizer → processing_class
    )

    print("✅ ORPO Trainer initialized (v0.12+ API)")
    print(f"  Beta (preference strength): {config.beta}")
    print(f"  Effective batch size: {config.batch_size * config.gradient_accumulation}")

# 5. LoRA 적용
from peft import LoraConfig, get_peft_model

if config.use_lora and not hasattr(model, 'peft_config'):
  lora_config = LoraConfig(
      r=32,
      lora_alpha=64,
      target_modules=["q_proj", "v_proj"],  # 타겟 줄이기
      lora_dropout=0.1,
  )
  model = get_peft_model(model, lora_config)
  model.print_trainable_parameters()

Map:   0%|          | 0/101314 [00:00<?, ? examples/s]

Map:   0%|          | 0/101314 [00:00<?, ? examples/s]

Map:   0%|          | 0/101314 [00:00<?, ? examples/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

✅ ORPO Trainer initialized (v0.12+ API)
  Beta (preference strength): 0.1
  Effective batch size: 16
trainable params: 25,165,824 || all params: 14,795,199,488 || trainable%: 0.1701


## 7. Alternative: DPO Training

In [ ]:
from trl import DPOConfig, DPOTrainer

if config.method == TrainingMethod.DPO:
    # DPO Configuration
    training_args = DPOConfig(
        output_dir=config.output_dir,

        # Training parameters
        num_train_epochs=config.num_epochs,
        per_device_train_batch_size=config.batch_size,
        per_device_eval_batch_size=config.batch_size,
        gradient_accumulation_steps=config.gradient_accumulation,

        # DPO specific
        beta=config.beta,
        learning_rate=config.learning_rate,

        # Performance
        bf16=True,
        gradient_checkpointing=True,

        # Other
        logging_steps=25,
        save_steps=500,
        warmup_ratio=0.1,
    )

    # Initialize DPO Trainer
    trainer = DPOTrainer(
        model=model,
        ref_model=None,  # Will create automatically
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=test_dataset,
        processing_class=tokenizer,  # TRL에서 tokenizer= 가 processing_class= 로 변경됨
    )

    print("✅ DPO Trainer initialized")

## 8. Start Training

In [ ]:
import time

print(f"🚀 Starting {config.method.value.upper()} training...")
print("="*50)
print(f"Model: {config.model_id}")
print(f"Dataset: {len(train_dataset):,} preference pairs")
print(f"Method: {config.method.value.upper()}")
print("="*50)

start_time = time.time()

# Train
train_result = trainer.train()

# Time
elapsed = (time.time() - start_time) / 60

print("="*50)
print(f"✅ Training complete!")
print(f"⏱️ Time: {elapsed:.1f} minutes")
print(f"📉 Final loss: {train_result.training_loss:.4f}")
print("="*50)

🚀 Starting ORPO training...
Model: Qwen/Qwen2.5-14B-Instruct
Dataset: 101,314 preference pairs
Method: ORPO


Step,Training Loss
25,3.084800
50,3.047700
75,2.994200
100,2.794100
125,2.563000
150,2.279700
175,1.909800
200,1.481400
225,1.102700
250,0.788300


Step,Training Loss
25,3.084800
50,3.047700
75,2.994200
100,2.794100
125,2.563000
150,2.279700
175,1.909800
200,1.481400
225,1.102700
250,0.788300


## 9. Save Model

In [ ]:
# Save model
print("💾 Saving model...")

save_path = f"{config.output_dir}/final"
trainer.save_model(save_path)
tokenizer.save_pretrained(save_path)

print(f"✅ Model saved to: {save_path}")

# If using LoRA, can merge
if config.use_lora:
    print("\n🔀 Merging LoRA weights...")
    model = model.merge_and_unload()
    model.save_pretrained(f"{config.output_dir}/merged")
    print(f"✅ Merged model saved")

## 10. Advanced Evaluation

In [ ]:
# Advanced testing with preference scoring
test_cases = [
    {
        "question": "Is TP53 upregulated in BRCA?",
        "good_answer": "No, TP53 is typically downregulated or mutated in breast cancer.",
        "bad_answer": "Yes, TP53 is upregulated."
    },
    {
        "question": "Which has higher expression in LIHC: MYC or EGFR?",
        "good_answer": "MYC typically shows higher expression in hepatocellular carcinoma.",
        "bad_answer": "I don't know."
    },
]

print("🧪 Evaluating model preferences:\n")

model.eval()

for i, test in enumerate(test_cases, 1):
    prompt = f"Question: {test['question']}\nAnswer:"

    # Generate response
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=100,
            do_sample=True,
            temperature=0.7,
            top_p=0.9,
        )

    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    response = response.split("Answer:")[-1].strip()

    print(f"Test {i}: {test['question']}")
    print(f"Model: {response}")
    print(f"Expected: {test['good_answer']}")
    print("-"*50)

## Summary

### 🎯 Advanced Training Complete!

**What we used:**
- ✅ **ORPO/DPO** - Latest preference optimization (better than GRPO)
- ✅ **Full Fine-tuning** - No quantization for best quality
- ✅ **BF16 Training** - Better than FP16 for stability
- ✅ **Flash Attention 2** - Faster and more memory efficient
- ✅ **Preference Learning** - Model learns what's right AND wrong

**Why this is better:**
1. **ORPO > GRPO**: More stable, better alignment
2. **No 4-bit**: Full precision = better quality
3. **Preference pairs**: Learns from both good and bad examples
4. **Latest methods**: Using 2024's best practices

**Results:**
- Model understands TCGA cancer genomics
- Can distinguish correct from incorrect answers
- Ready for production deployment